In [10]:
# ================================================================
# CREDIBILITY MODEL CELL 1 — Imports
# ================================================================
import pickle
import re
import requests
from bs4 import BeautifulSoup
from textblob import TextBlob
from urllib.parse import urlparse

print('Imports done!')

Imports done!


In [11]:
# ================================================================
# CREDIBILITY MODEL CELL 2 — Load Model
# ================================================================
vectorizer = pickle.load(open(r'C:\Users\wtpir\Documents\AI fake news detection\news\AI\Text Models\Text Model\tfidf_vectorizer.pkl', 'rb'))
best_model = pickle.load(open(r'C:\Users\wtpir\Documents\AI fake news detection\news\AI\Text Models\Text Model\best_fake_news_model.pkl', 'rb'))

print('Model loaded!')

Model loaded!


In [12]:
# ================================================================
# CREDIBILITY MODEL CELL 3 — All Helper Functions
# ================================================================

# ---------- Text Prediction ----------
def predict_news(text, confidence_threshold=0.40):
    text = re.sub(r'^[A-Z\s,\.]+\(Reuters\)\s*[-\u2013]\s*', '', str(text))
    text_tfidf = vectorizer.transform([text])
    if text_tfidf.nnz == 0:
        return 'Uncertain', 50
    prob = best_model.predict_proba(text_tfidf)[0]
    pred = best_model.predict(text_tfidf)[0]
    confidence = prob[pred]
    if confidence < confidence_threshold:
        return 'Uncertain', 50
    label = 'Real' if pred == 1 else 'Fake'
    score = int(confidence * 100) if pred == 1 else int((1 - confidence) * 100)
    return label, score


# ---------- Sentiment ----------
SENSATIONAL_WORDS = [
    'breaking', 'shocking', 'bombshell', 'explosive', 'urgent',
    'alert', 'exposed', 'leaked', 'confirmed', 'proof', 'secret',
    'censored', 'banned', 'deleted', 'share', 'wake up', 'sheeple',
    'deep state', 'cover up', 'mainstream media', 'they dont want',
    'before they delete', 'massive', 'disgusting', 'unbelievable'
]

def get_sentiment_score(text):
    blob = TextBlob(text)
    subjectivity = blob.sentiment.subjectivity
    text_lower = text.lower()
    found_words = [w for w in SENSATIONAL_WORDS if w in text_lower]
    exclamations = text.count('!')
    caps_words = [w for w in text.split() if w.isupper() and len(w) > 2]
    sensational_score = min(
        len(found_words) * 10 +
        exclamations * 5 +
        len(caps_words) * 5,
        100
    )
    # Combined — higher = more fake indicators
    fake_indicator = int(subjectivity * 40 + (sensational_score / 100) * 60)
    return fake_indicator, round(subjectivity, 2), sensational_score


# ---------- Source Credibility ----------
TRUSTED_SOURCES = [
    'reuters.com', 'bbc.com', 'bbc.co.uk', 'apnews.com',
    'npr.org', 'theguardian.com', 'nytimes.com', 'washingtonpost.com',
    'bloomberg.com', 'wsj.com', 'time.com', 'politico.com',
    'cbsnews.com', 'nbcnews.com', 'abcnews.go.com', 'cnn.com',
    'aljazeera.com', 'dw.com', 'france24.com', 'euronews.com'
]
FAKE_SOURCES = [
    'infowars.com', 'naturalnews.com', 'breitbart.com',
    'beforeitsnews.com', 'yournewswire.com', 'worldnewsdailyreport.com',
    'nationalreport.net', 'abcnews.com.co', 'cnn.com.de'
]

def get_source_score(url):
    if not url:
        return 50, 'No URL provided ⚠️'
    try:
        domain = urlparse(url).netloc.replace('www.', '')
        for s in TRUSTED_SOURCES:
            if s in domain:
                return 100, f'Trusted ✅ ({domain})'
        for s in FAKE_SOURCES:
            if s in domain:
                return 0, f'Unreliable ❌ ({domain})'
        return 40, f'Unknown ⚠️ ({domain})'
    except:
        return 50, 'Could not check source ⚠️'


# ---------- URL Scraper ----------
def scrape_url(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()
        article = soup.find('article')
        if article:
            text = article.get_text(separator=' ')
        else:
            text = ' '.join([p.get_text() for p in soup.find_all('p')])
        return ' '.join(text.split())
    except:
        return None

print('All helper functions ready!')

All helper functions ready!


In [13]:
# ================================================================
# CREDIBILITY MODEL CELL 4 — Main Credibility Scorer
# ================================================================
def credibility_score(text=None, url=None):
    print('=' * 60)
    print('         CREDIBILITY SCORER — FULL ANALYSIS')
    print('=' * 60)

    # ── Step 1: Get text ──────────────────────────────────────
    if url and not text:
        print('🌐 Fetching article from URL...')
        text = scrape_url(url)
        if not text:
            print('❌ Could not extract text from URL')
            return

    if not text:
        print('❌ No text provided')
        return

    # ── Step 2: Text Model ────────────────────────────────────
    text_label, text_confidence = predict_news(text)
    if text_label == 'Real':
        text_score = text_confidence          # high = credible
    elif text_label == 'Fake':
        text_score = 100 - text_confidence    # low = not credible
    else:
        text_score = 50

    print(f'\n📄 TEXT ANALYSIS:')
    print(f'   Prediction:  {text_label}')
    print(f'   Confidence:  {text_confidence}%')
    print(f'   Text Score:  {text_score}/100')

    # ── Step 3: Sentiment ─────────────────────────────────────
    fake_indicator, subjectivity, sensational = get_sentiment_score(text)
    sentiment_score = 100 - fake_indicator   # flip — high = credible

    print(f'\n😡 SENTIMENT ANALYSIS:')
    print(f'   Subjectivity:    {subjectivity} (0=fact, 1=opinion)')
    print(f'   Sensationalism:  {sensational}/100')
    print(f'   Sentiment Score: {sentiment_score}/100')

    # ── Step 4: Source ────────────────────────────────────────
    source_score, source_label = get_source_score(url)

    print(f'\n🌐 SOURCE CHECK:')
    print(f'   Source: {source_label}')
    print(f'   Score:  {source_score}/100')

    # ── Step 5: Combined Score ────────────────────────────────
    # Weights: Text 50% + Sentiment 30% + Source 20%
    if url:
        combined = int(
            text_score     * 0.50 +
            sentiment_score * 0.30 +
            source_score   * 0.20
        )
    else:
        # No URL — redistribute source weight
        combined = int(
            text_score      * 0.60 +
            sentiment_score * 0.40
        )

    # ── Step 6: Final Verdict ─────────────────────────────────
    if combined >= 70:
        verdict      = 'Real 🟢'
        verdict_note = 'Likely Credible'
        bar_color    = '🟩'
    elif combined >= 40:
        verdict      = 'Uncertain ⚠️'
        verdict_note = 'Treat with Caution'
        bar_color    = '🟨'
    else:
        verdict      = 'Fake 🔴'
        verdict_note = 'Likely Not Credible'
        bar_color    = '🟥'

    # Progress bar
    filled = int(combined / 10)
    empty  = 10 - filled
    bar    = bar_color * filled + '⬜' * empty

    print(f'\n{"=" * 60}')
    print(f'  CREDIBILITY SCORE: {combined}/100')
    print(f'  {bar}')
    print(f'  FINAL VERDICT:     {verdict}')
    print(f'  NOTE:              {verdict_note}')
    print(f'{"=" * 60}')
    print(f'\n  Breakdown:')
    print(f'  📄 Text Model  (50%): {text_score}/100')
    print(f'  😡 Sentiment   (30%): {sentiment_score}/100')
    print(f'  🌐 Source      (20%): {source_score}/100')
    print(f'{"=" * 60}')

    return {
        'text_score'     : text_score,
        'sentiment_score': sentiment_score,
        'source_score'   : source_score,
        'combined_score' : combined,
        'verdict'        : verdict,
        'text_label'     : text_label,
        'source_label'   : source_label
    }

print('credibility_score() is ready!')

credibility_score() is ready!


In [14]:
# ================================================================
# CREDIBILITY MODEL CELL 5 — Test with text only
# ================================================================
fake_text = "BREAKING!! Hillary Clinton ARRESTED by FBI!! Sources confirm Obama involved in MASSIVE COVER-UP!! Share before they DELETE THIS!!"

print('--- FAKE NEWS TEST ---')
credibility_score(text=fake_text)

--- FAKE NEWS TEST ---
         CREDIBILITY SCORER — FULL ANALYSIS

📄 TEXT ANALYSIS:
   Prediction:  Fake
   Confidence:  1%
   Text Score:  99/100

😡 SENTIMENT ANALYSIS:
   Subjectivity:    1.0 (0=fact, 1=opinion)
   Sensationalism:  100/100
   Sentiment Score: 0/100

🌐 SOURCE CHECK:
   Source: No URL provided ⚠️
   Score:  50/100

  CREDIBILITY SCORE: 59/100
  🟨🟨🟨🟨🟨⬜⬜⬜⬜⬜
  FINAL VERDICT:     Uncertain ⚠️
  NOTE:              Treat with Caution

  Breakdown:
  📄 Text Model  (50%): 99/100
  😡 Sentiment   (30%): 0/100
  🌐 Source      (20%): 50/100


{'text_score': 99,
 'sentiment_score': 0,
 'source_score': 50,
 'combined_score': 59,
 'verdict': 'Uncertain ⚠️',
 'text_label': 'Fake',
 'source_label': 'No URL provided ⚠️'}

In [15]:
# ================================================================
# CREDIBILITY MODEL CELL 6 — Test with text only
# ================================================================
real_text = "The Federal Reserve raised interest rates by 0.25 percentage points on Wednesday, citing continued progress toward its inflation target."

print('--- REAL NEWS TEST ---')
credibility_score(text=real_text)

--- REAL NEWS TEST ---
         CREDIBILITY SCORER — FULL ANALYSIS

📄 TEXT ANALYSIS:
   Prediction:  Real
   Confidence:  82%
   Text Score:  82/100

😡 SENTIMENT ANALYSIS:
   Subjectivity:    0.0 (0=fact, 1=opinion)
   Sensationalism:  0/100
   Sentiment Score: 100/100

🌐 SOURCE CHECK:
   Source: No URL provided ⚠️
   Score:  50/100

  CREDIBILITY SCORE: 89/100
  🟩🟩🟩🟩🟩🟩🟩🟩⬜⬜
  FINAL VERDICT:     Real 🟢
  NOTE:              Likely Credible

  Breakdown:
  📄 Text Model  (50%): 82/100
  😡 Sentiment   (30%): 100/100
  🌐 Source      (20%): 50/100


{'text_score': 82,
 'sentiment_score': 100,
 'source_score': 50,
 'combined_score': 89,
 'verdict': 'Real 🟢',
 'text_label': 'Real',
 'source_label': 'No URL provided ⚠️'}

In [16]:
# ================================================================
# CREDIBILITY MODEL CELL 7 — Test with URL
# ================================================================
url = "https://www.reuters.com/world/us/senate-passes-government-funding-bill-2024-03-23/"

print('--- URL TEST ---')
credibility_score(url=url)

--- URL TEST ---
         CREDIBILITY SCORER — FULL ANALYSIS
🌐 Fetching article from URL...

📄 TEXT ANALYSIS:
   Prediction:  Fake
   Confidence:  12%
   Text Score:  88/100

😡 SENTIMENT ANALYSIS:
   Subjectivity:    0.0 (0=fact, 1=opinion)
   Sensationalism:  0/100
   Sentiment Score: 100/100

🌐 SOURCE CHECK:
   Source: Trusted ✅ (reuters.com)
   Score:  100/100

  CREDIBILITY SCORE: 94/100
  🟩🟩🟩🟩🟩🟩🟩🟩🟩⬜
  FINAL VERDICT:     Real 🟢
  NOTE:              Likely Credible

  Breakdown:
  📄 Text Model  (50%): 88/100
  😡 Sentiment   (30%): 100/100
  🌐 Source      (20%): 100/100


{'text_score': 88,
 'sentiment_score': 100,
 'source_score': 100,
 'combined_score': 94,
 'verdict': 'Real 🟢',
 'text_label': 'Fake',
 'source_label': 'Trusted ✅ (reuters.com)'}